#  **Python Class 18 – Object-Oriented Programming (OOP) Deep Dive**

In the previous class, we introduced the foundations of **Object-Oriented Programming (OOP)**. You learned how to define classes, create objects, use instance and class attributes, and apply inheritance—including multiple inheritance—to reuse and extend code.

In today’s class, we will **go deeper into OOP**, focusing on how Python actually resolves behavior in complex class hierarchies and how professional Python code is designed.

---

## What We Will Learn Today

###  Method Resolution Order (MRO)

We will explore how Python decides **which method to call** when multiple classes define the same method, and how the **C3 linearization algorithm** keeps inheritance predictable and safe.

---

###  The `super()` Function

You will learn how `super()` works internally, why it is essential in multiple inheritance, and how it enables cooperative method calls across class hierarchies.

---

###  Polymorphism

We will see how different objects can respond to the **same method name** in different ways, enabling flexible and extensible designs.
---

###  Encapsulation

We will discuss:

* public vs protected vs private attributes
* name mangling (`__attr`)
* how encapsulation helps prevent accidental misuse of objects

---

##  Why This Class Matters

These concepts are critical for:

* writing maintainable and scalable code
* understanding real-world Python libraries (NumPy, Pandas, PyTorch)
* designing clean APIs and reusable components
* mastering advanced OOP interviews and production code

By the end of this lesson, you will not only *use* classes—you will understand how Python **thinks about classes**.



# Method Resolution Order (MRO) & super()

When a class inherits from multiple parents, Python needs a `GPS` to decide which method to run. This is where the `C3 Linearization` algorithm comes in.

The `Diamond Problem` Example
Let's look at the classic diamond shape. If every class has a method called `show()`, which one runs?

In [1]:
class A:
    def show(self):
        print("Process starting in A", end = '\n\n')

class B(A):
    def show(self):
        print("Doing work in B", end = '\n\n')
        super().show()

class C(A):
    def show(self):
        print("Doing work in C", end = '\n\n')
        super().show()

class D(B, C):
    def show(self):
        print("Starting in D", end = '\n\n')
        super().show()

# creating instance of D
obj = D()
obj.show()

Starting in D

Doing work in B

Doing work in C

Process starting in A



In [2]:
print(D.mro(), end = '\n\n')
print(D.__mro__, end = '\n\n')

[<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>]

(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)



**The MRO Chain**: Python looks for the method in the order: `D -> B -> C -> A -> object`.

`super()` is not **Parent**: A common mistake is thinking `super()` always calls the parent. In class B, `super()` actually points to C in this specific hierarchy! This is called **Cooperative Multiple Inheritance**.

### C3 Linearization

To understand **C3 Linearization**, you first have to understand the problem it solves. In complex inheritance (especially Multiple Inheritance), Python needs a consistent way to flatten a **tree** or **graph** of classes into a single, linear list. This list is the **Method Resolution Order (MRO)**.

C3 Linearization is the specific algorithm Python uses to ensure this list is **monotonic** (it doesn't flip-flop the order of classes) and respects **local precedence** (parents are checked before grandparents).

---

## The Core Rule of C3

C3 follows a simple formula for any class C:

$$L(C) = [C] + \text{merge}(L(P_1), L(P_2), \dots, L(P_n), [P_1, P_2, \dots, P_n])$$

Where:

* L(C) is the linear order of class C.
* $P_1, P_2$ are the parent classes in the order they were defined.
* The **merge** step is the "secret sauce" that picks the next class for the list.

---

## How the **Merge** Logic Works

The merge looks at the heads (the first item) of the parent lists. It picks a head if:

1. It is the first item in a list.
2. It **does not** appear anywhere else in the **tail** (the rest) of the other lists.

If a class is in the tail of another list, it means that class is a **parent** elsewhere and must wait its turn. This prevents a grandparent from being called before a parent.

---

## A Step-by-Step Example

Let’s look at a simple hierarchy:

```python
class O: pass
class A(O): pass
class B(O): pass
class C(A, B): pass

```

### 1. Identify the Basics

* L(O) = [O]
* L(A) = [A, O]
* L(B) = [B, O]

### 2. Calculate L(C)

Formula: $L(C) = [C] + \text{merge}(L(A), L(B), [A, B])$


**Step A: Put in the heads**

* $L(C) = [C] + \text{merge}([A, O], [B, O], [A, B])$



**Step B: Can we pick A?**

* Is A at the head of a list? **Yes** ([A, O]).

* Is A in the tail of any other list? **No**. (It's at the head of [A, B], which is fine).

* *Result:* Pick **A**.
$L(C) = [C, A] + \text{merge}([O], [B, O], [B])$


**Step C: Can we pick O?**
* Is O at the head? **Yes**.

* Is O in the tail of another list? **Yes**! It is in the tail of [B, O].
*Result:* **Skip O** for now.

* **Step D: Can we pick B?**
Is B at the head? **Yes** ([B, O]).

* Is B in the tail of another list? **No**.
*Result:* Pick **B**.

* $L(C) = [C, A, B] + \text{merge}([O], [O])$

**Step E: Pick O**
* Finally, pick O as it’s the only one left.


**Final MRO:** `[C, A, B, O]`

---

## Why does this matter for you?

Without C3, you could end up with the **"Inconsistent Hierarchy" error**. If a developer tries to create a class structure that violates these logical rules, Python will raise a `TypeError: Cannot create a consistent method resolution order`.

It ensures that:

1. **Children come before Parents.**
2. **Left-hand parents come before Right-hand parents.**
3. **No class is searched twice.**


##The "Impossible" Inheritance Code

In [3]:
class A: pass
class B: pass

# X says: A comes before B
class X(A, B): pass

# Y says: B comes before A
class Y(B, A): pass

# Z says: I want to inherit from X AND Y
try:
    class Z(X, Y): pass
except TypeError as e:
    print(f"CRITICAL ERROR: {e}")

CRITICAL ERROR: Cannot create a consistent method resolution
order (MRO) for bases A, B


###Why did Python throw a `TypeError`?

When Python tries to build the `MRO` for `Z`, it looks at its immediate parents: `X` and `Y`.

* From `X`: Python is told: "`A` must come before `B`."

* From `Y`: Python is told: "`B` must come before `A`."

* The Conflict: Python reaches a **deadlock**. It cannot satisfy both conditions at the same time.

In technical terms, this violates **Monotonicity**. If Class $X$ says $A < B$, then any subclass of $X$ must also respect that $A < B$. Since `Z` is a subclass of both $X$ and $Y$, and they have opposite rules, the C3 Linearization algorithm fails.

###How to Fix It

To fix this, the developer must decide on a consistent hierarchy. Usually, this means making sure that if multiple classes share parents, they list them in the same order.

In [4]:
# Fixed version: Both agree on (A, B)
class X(A, B): pass
class Y(A, B): pass
class Z(X, Y): pass

##Moving to `super()`

Now we should try to understand how Python finds the next class in the list, we can explain `super()`.

A common misconception is that `super()` means **call my parent.** Correct Definition: `super()` means **call the next class in the MRO list.**

Example of `super()` in action:

In [5]:
class Base:
    def __init__(self):
        print("Base reached", end = '\n\n')

class Left(Base):
    def __init__(self):
        print("Entering Left", end = '\n\n')
        super().__init__()
        print("Exiting Left", end = '\n\n')

class Right(Base):
    def __init__(self):
        print("Entering Right", end = '\n\n')
        super().__init__()
        print("Exiting Right", end = '\n\n')

class Combined(Left, Right):
    def __init__(self):
        print("Entering Combined", end = '\n\n')
        super().__init__()
        print("Exiting Combined", end = '\n\n')

c = Combined()

Entering Combined

Entering Left

Entering Right

Base reached

Exiting Right

Exiting Left

Exiting Combined



**Cooperative Multiple Inheritance** is the "teamwork" version of programming. In many languages, multiple inheritance is messy because parents don't know about each other. In Python, `super()` acts as a relay baton, passing the execution to the next class in line.

###Breaking Down the Execution Flow
Let's look at the output of the code from the previous step. If we run `c = Combined()`, here is the "Play-by-Play":

| Step | Current Class | Action            | Why                                                  |
| ---: | ------------- | ----------------- | ---------------------------------------------------- |
|    1 | `Combined`    | Enters `__init__` | We instantiated `Combined()`                         |
|    2 | `Combined`    | Calls `super()`   | According to MRO, the next class is `Left`           |
|    3 | `Left`        | Enters `__init__` | Prints **"Entering Left"**                           |
|    4 | `Left`        | Calls `super()`   | In this MRO, the next class is `Right` (not `Base`)  |
|    5 | `Right`       | Enters `__init__` | Prints **"Entering Right"**                          |
|    6 | `Right`       | Calls `super()`   | The next class in MRO is `Base`                      |
|    7 | `Base`        | Enters and exits  | `Base` has no `super()`, so it finishes immediately  |
|    8 | `Right`       | Exits `__init__`  | Control returns to `Right` to complete its execution |
|    9 | `Left`        | Exits `__init__`  | Control returns to `Left`                            |
|   10 | `Combined`    | Exits `__init__`  | Initialization is fully complete                     |


The Resulting Output:
```text
Entering Combined
Entering Left
Entering Right
Base reached
Exiting Right
Exiting Left
Exiting Combined
```

###Two Rules for "Cooperation"

For this to work in production code, your students must follow two golden rules:

* **Use `super()` everywhere:** If `Left` forgot to call `super()`, the chain would break. `Right` and `Base` would never be initialized, even though `Combined` inherited from them!

* **Matching Signatures:** All methods in the chain should ideally accept the same arguments (usually handled via `*args` and `**kwargs`) so the baton can be passed without a "Size Mismatch" error.

Why this matters: This is how frameworks like Django (Mixins) or Kivy work. They allow you to "plug in" features (like a SearchMixin and a LoggingMixin) and ensure both initialize correctly without you having to manually call each parent class name.

# Polymorphism

**Polymorphism** is one of the most powerful concepts in OOP. Derived from the Greek words for **many forms,** it allows different classes to be treated as instances of the same general category through a common interface.

In Python, this means that different objects can have methods with the **same name** but **different behaviors**.

The meaning of polymorphism is the condition of occurrence in different forms.

Polymorphism is a very important concept in programming. It refers to the use of a single type entity (method, operator or object) to represent different types in different scenarios.

Let's take an example:

## Polymorphism in addition operator
We know that the `+` operator is used extensively in Python programs. But, it does not have a single usage.

For integer data types, `+` operator is used to perform arithmetic addition operation.

In [6]:
v1 = 78
v2 = 57
print(v1 + v2)

135


Similarly, for string data types, `+` operator is used to perform concatenation.

In [7]:
s1 = "Rafo"
s2 = "Arakelyan"
print(s1 + s2)

RafoArakelyan


Here, we can see that a single operator `+` has been used to carry out different operations for distinct data types. This is one of the most simple occurrences of polymorphism in Python.

## Function Polymorphism in Python

There are some functions in Python which are compatible to run with multiple data types.

One such function is the len() function. It can run with many data types in Python. Let's look at some example use cases of the function.

## Polymorphic len

In [8]:
print(len("Բուքն է լալիս. հողմ ու ձյուն,"))

print(len({"«Գիշեր»": "Երգում է քամին, լալիս է նորից,",
           "«Անծանոթ աղջկան»": "Լույսն էր մեռնում, օրը մթնում."}))

print(len(["Վահան", "Տերյան", "«Ուշացած սեր»"]))

29
2
3


Here, we can see that many data types such as string, list, tuple, set, and dictionary can work with the len() function. However, we can see that it returns specific information about specific data types.

![](https://cdn.programiz.com/sites/tutorial2program/files/func-polymorphism.png)

## Class Polymorphism in Python

Polymorphism is a very important concept in Object-Oriented Programming.

We can use the concept of polymorphism while creating class methods as Python allows different classes to have methods with the same name.

We can then later generalize calling these methods by disregarding the object we are working with. Let's look at an example:

## Polymorphism in Class Methods

In [9]:
class Cat:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def info(self):
        print(f"I am a cat. My name is {self.name}.\
                I am {self.age} years old.")

    def make_sound(self):
        print("Meow")


class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def info(self):
        print(f"I am a dog. My name is {self.name}.\
                I am {self.age} years old.")

    def make_sound(self):
        print("Bark")


In [10]:
cat1 = Cat("Kitty", 2.5)
dog1 = Dog("Fluffy", 4)

for animal in (cat1, dog1):
    animal.make_sound()
    animal.info()
    animal.make_sound()

Meow
I am a cat. My name is Kitty.                I am 2.5 years old.
Meow
Bark
I am a dog. My name is Fluffy.                I am 4 years old.
Bark


Here, we have created two classes Cat and Dog. They share a similar structure and have the same method names info() and make_sound().

However, notice that we have not created a common superclass or linked the classes together in any way. Even then, we can pack these two different objects into a tuple and iterate through it using a common animal variable. It is possible due to polymorphism

# Polymorphism and Inheritance

Like in other programming languages, the child classes in Python also inherit methods and attributes from the parent class. We can redefine certain methods and attributes specifically to fit the child class, which is known as Method Overriding.

Polymorphism allows us to access these overridden methods and attributes that have the same name as the parent class.


###Polymorphism via Method Overriding
The most common way students encounter polymorphism is through inheritance. A child class "overrides" a method defined in the parent class to provide its own specific implementation.

In [11]:
class Animal:
    def speak(self):
        raise NotImplementedError("Subclass must implement abstract method")

class Dog(Animal):
    def speak(self):
        return "Woof!"

class Cat(Animal):
    def speak(self):
        return "Meow!"

# A function that doesn't care WHAT animal it gets,
# as long as it has a .speak() method.
def make_it_talk(animal):
    print(animal.speak())

animals = [Dog(), Cat()]

for a in animals:
    make_it_talk(a)

Woof!
Meow!


**Key Lesson:** The function make_it_talk is polymorphic. It works with any object that follows the Animal interface.

### "Duck Typing" (Python’s Special Flavor)
In strictly typed languages (like Java or C++), you must explicitly inherit from a class to achieve polymorphism. In Python, we use Duck Typing:

    "If it walks like a duck and quacks like a duck, then it must be a duck."

In Python, if an object has the required method, Python will run it—regardless of the object's class hierarchy.

In [12]:
class Robot:
    def speak(self):
        return "Beep Boop!"

# Robot does NOT inherit from Animal, but...
make_it_talk(Robot()) # This works perfectly!

Beep Boop!


## Another Method Overriding example

In [13]:
from math import pi

class Shape:
    def __init__(self, name):
        self.name = name

    def area(self):
        pass

    def fact(self):
        return "I am a two-dimensional shape."

    def __str__(self):
        return self.name


class Square(Shape):
    def __init__(self, length):
        super().__init__("Square") # Shape.__init__()
        self.length = length

    def area(self):
        return self.length**2

    def fact(self):
        return "Squares have each angle equal to 90 degrees."


class Circle(Shape):
    def __init__(self, radius):
        super().__init__("Circle") # self.name = "Circle"
        self.radius = radius

    def area(self):
        return pi*self.radius**2


a = Square(4)
b = Circle(7)

In [14]:
print(b, end = '\n\n')
print(b.fact(), end = '\n\n')
print(a.fact(), end = '\n\n')
print(b.area())

Circle

I am a two-dimensional shape.

Squares have each angle equal to 90 degrees.

153.93804002589985


Here, we can see that the methods such as __str__(), which have not been overridden in the child classes, are used from the parent class.

Due to polymorphism, the Python interpreter automatically recognizes that the fact() method for object a(Square class) is overridden. So, it uses the one defined in the child class.

On the other hand, since the fact() method for object b isn't overridden, it is used from the Parent Shape class.

![](https://cdn.programiz.com/sites/tutorial2program/files/python-polymorphism.png)

## Operator Overloading

Python operators work for built-in classes. But the same operator behaves differently with different types. For example, the `+` operator will perform arithmetic addition on two numbers, merge two lists, or concatenate two strings.

This feature in Python that allows the same operator to have different meaning according to the context is called operator overloading.

So what happens when we use them with objects of a user-defined class? Let us consider the following class, which tries to simulate a point in 2-D coordinate system.


In [15]:
class Point:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __str__(self):
        return "({0},{1})".format(self.x, self.y)

    def __add__(self, other):
       #return self.x + self.y
        x = self.x + other.x
        y = self.y + other.y
        print(self)
        return Point(x, y)

In [16]:
p1 = Point(78, 88)
p2 = Point(2, 111)
print(p1+p2)

(78,88)
(80,199)


In [17]:
p1 = Point(78, 88)
p2 = Point(2, 111)
print(p1+p2)

(78,88)
(80,199)


The reason you got an error is that the `+` operator is a **binary operator**—it requires two sides to work (a left side and a right side).

###How Python translates the `+` sign

When you write `p1 + p2`, Python internally converts that code into a method call. It looks like this:

`p1 + p2` $\rightarrow$ `p1.__add__(p2)`

* `self` is the object on the left (`p1`).

* `other` is the object on the right (`p2`).

If your method is defined as `def __add__(self):`, Python tries to pass `p2` into it, but the method isn't built to catch it. This results in a `TypeError: __add__()` takes 1 positional argument but 2 were given.

Here, we can see that a TypeError was raised, since Python didn't know how to add two Point objects together.

However, we can achieve this task in Python through operator overloading. But first, let's get a notion about special functions.

## Python Special Functions

Class functions that begin with double underscore __ are called special functions in Python.

These functions are not the typical functions that we define for a class. The __init__() function we defined above is one of them. It gets called every time we create a new object of that class.

There are numerous other special functions in Python. Visit Python Special Functions to learn more about them.

Using special functions, we can make our class compatible with built-in functions.

## How `print()` works

In [18]:
p1 = Point(2,3)
print(p1)

(2,3)


Suppose we want the print() function to print the coordinates of the Point object instead of what we got. We can define a __str__() method in our class that controls how the object gets printed. Let's look at how we can achieve this:

In [19]:
class Point:
    def __init__(self, x = 0, y = 0):
        self.x = x
        self.y = y

    def __str__(self):
        return "({0},{1})".format(self.x, self.y)

Now let's try the print() function again.

In [20]:
p1 = Point(2, 3)
print(p1)

(2,3)


That's better. Turns out, that this same method is invoked when we use the built-in function str() or format().

So, when you use str(p1) or format(p1), Python internally calls the p1.__str__() method. Hence the name, special functions.

Now let's go back to operator overloading.

## Overloading the `+` Operator

To overload the `+` operator, we will need to implement `__add__()` function in the class. With great power comes great responsibility. We can do whatever we like, inside this function. But it is more sensible to return a Point object of the coordinate sum.

In [21]:
class Point:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __str__(self):
        return "({0},{1})".format(self.x, self.y)

    def __add__(self, other):
        x = self.x + other.x
        y = self.y + other.y
        print(other)
        return Point(x, y)

Now let's try the addition operation again:

In [22]:
p1 = Point(1, 2)
p2 = Point(2, 3)

print(p1+p2)

(2,3)
(3,5)


In [23]:
class Point:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __str__(self):
        return "({0},{1})".format(self.x, self.y)

    def __add__(self, other):
        x = self.x + other.x
        y = self.y + other.y
        return Point(x, y)

    def __sub__(self, other):
        x = self.x - other.x
        y = self.y - other.y
        return Point(x, y)



In [24]:
p1 = Point(1, 2)
p2 = Point(2, 3)

print(p1 - p2)

(-1,-1)


What actually happens is that, when you use p1 + p2, Python calls p1.__add__(p2) which in turn is Point.__add__(p1,p2). After this, the addition operation is carried out the way we specified.

Similarly, we can overload other operators as well. The special function that we need to implement is tabulated below.

Operator |	Expression	| Internally |
--- | ----| ----|
Addition	| p1 + p2	| p1.__add__(p2) |
Subtraction	| p1 - p2	| p1.__sub__(p2) |
Multiplication	|  p1 * p2	|p1.__mul__(p2) |
Power	| p1 ** p2	| p1.__pow__(p2) |
Division	| p1 / p2	| p1.__truediv__(p2) |
Floor Division |	p1 // p2	| p1.__floordiv__(p2) |
Remainder (modulo)	| p1 % p2	| p1.__mod__(p2) |

## Overloading Comparison Operators

Python does not limit operator overloading to arithmetic operators only. We can overload comparison operators as well.

Suppose we wanted to implement the less than symbol < symbol in our Point class.

Let us compare the magnitude of these Points from the origin and return the result for this purpose. It can be implemented as follows.

In [25]:
# overloading the less than operator
class Point:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __str__(self):
        return "({0},{1})".format(self.x, self.y)

    def __lt__(self, other):
        self_mag = (self.x ** 2) + (self.y ** 2)
        other_mag = (other.x ** 2) + (other.y ** 2)
        return self_mag < other_mag

p1 = Point(1,1)
p2 = Point(-2,-3)
p3 = Point(1,-1)

# use less than
print(p1<p2, end = '\n\n')
print(p2<p3, end = '\n\n')
print(p1<p3)

True

False

False


Similarly, the special functions that we need to implement, to overload other comparison operators are tabulated below.

Operator |	Expression | Internally |
--- | ---- | ---- |
Less than	| p1 < p2	| p1.__lt__(p2) |
Less than or equal to	| p1 <= p2 |	p1.__le__(p2) |
Equal to	| p1 == p2 |	p1.__eq__(p2)
Not equal to	| p1 != p2 |	p1.__ne__(p2) |
Greater than	| p1 > p2	| p1.__gt__(p2) |
Greater than or equal to	| p1 >= p2 | 	p1.__ge__(p2) |

In [26]:
[1, 2, 3] + [5, 4, 1]

[1, 2, 3, 5, 4, 1]

In [27]:
class CustumList(list):
  def __add__(self, other):
    if len(self) == len(other):
      out = []
      for i in range(len(self)):
        out.append(self[i] + other[i])
      return out
    return super().__add__(other)

In [28]:
a = CustumList([1, 2, 3])
b = CustumList([5, 4, 1])
a + b

[6, 6, 4]

# Encapsulation

Using OOP in Python, we can restrict access to methods and variables. This prevents data from direct modification which is called **encapsulation**. **Encapsulation** is the practice of bundling data (attributes) and methods (functions) into a single unit (a class) and **restricting direct access** to some of those components.

Think of it like a **medical capsule**: the medicine (data) is inside, and you can only interact with it through the outer shell (the interface).

In Python, we don't have "hard" private keywords like Java or C++. Instead, we use naming conventions to tell other developers how to treat our data. We denote private attributes using underscore as the prefix i.e single `_` or double `__`.

## Python access modifiers: public, private and protected

### The Three Levels of Access

Python uses underscores to signal the "privacy" level of an attribute:

| Level     | Syntax           | Meaning                                                                 |
| --------- | ---------------- | ----------------------------------------------------------------------- |
| Public    | `self.balance`   | Accessible from anywhere                                                |
| Protected | `self._balance`  | “Please don’t touch this unless you are a subclass” *(convention only)* |
| Private   | `self.__balance` | “I really don’t want you to access this” *(triggers name mangling)*     |





* **Private** members of a class are denied access from the environment outside the class. They can be handled only from within the class.

* **Public** members (generally methods declared in a class) are accessible from outside the class.  From other side **public** methods are needed to access **private** members.

* **Protected** members of a class are accessible from within the class and are also available to its sub-classes.

**Python doesn't have any mechanism that effectively restricts access to any instance variable or method. Python prescribes a convention of prefixing the name of the variable/method with single or double underscore to emulate the behaviour of protected and private access specifiers.**

All members in a Python class are public by default. Any member can be accessed from outside the class environment.

### Protected attributes
Python's convention to make an instance variable protected is to add a prefix `_` (single underscore) to it.


In [29]:
class Employee:
  def __init__(self, name, sal):
    self._name = name  # protected attribute
    self._salary = sal # protected attribute

In fact, this doesn't prevent instance variables from accessing or modifying the instance.

In [30]:
emp2 = Employee("Hakob", 72500)
# access the protected attribute
print(emp2._salary)

72500


In [31]:
# modify the protected attribute
emp2._salary = 250000
print(emp2._salary)

250000


Hence, the responsible programmer would refrain from accessing and modifying instance variables prefixed with `_` from outside its class.

### Private attributes
A double underscore `__` prefixed to a variable makes it **private**. It gives a strong suggestion not to touch it from outside the class. Any attempt to do so will result in an **AttributeError**:

In [32]:
class Employee:
  def __init__(self, name, sal):
    self.__name = name  # private attribute
    self.__salary = sal # private attribute

  def set_salary(self, new_salary):
    self.__salary = new_salary

  def get_salary(self):
    return self.__salary

In [33]:
emp3 = Employee("Hakob", 100000) # again Hakob
emp3.__salary

AttributeError: 'Employee' object has no attribute '__salary'

In [34]:
print(emp3.get_salary())

100000


In [35]:
# emp3.__salary = 111000
emp3.set_salary(111000)
print(emp3.get_salary())

111000


Python performs name mangling of private variables. Every member with double underscore will be changed to **object._class__variable**. If so required, it can still be accessed from outside the class, but the practice should be refrained.

In [36]:
emp4 = Employee("Hakob", 10000)
emp4._Employee__salary

10000

In [37]:
emp4._Employee__salary = 20000
emp4._Employee__salary
#emp4.get_salary()

20000

## Naming conventions and  Mangling
Single and double underscores have a meaning in Python variable and method names. Some of that meaning is merely by convention and intended as a hint to the programmer — and some of it is enforced by the Python interpreter.

* Single Leading Underscore: `_var`
* Single Trailing Underscore: `var_`
* Double Leading Underscore: `__var`
* Double Leading and Trailing Underscore: `__var__`
* Single Underscore: `_`

### Single Leading Underscore: `_var`

Single underscores are a Python naming convention indicating a name is meant for internal use. It is generally not enforced by the Python interpreter and meant as a hint to the programmer only.

It’s a hint to the programmer — and it means what the Python community agrees it should mean, but it does not affect the behavior of your programs.

In [38]:
class Test:
  def __init__(self):
    self.foo = 11
    self._bar = 23

  def _some_method(self):
    return 0

**However, leading underscores do impact how names get imported from modules. Imagine you had the following code in a module called `my_module.py`:**


In [39]:
# Do not run
# This is my_module.py:

def external_func():
    return 23

def _internal_func():
    return 42

In [40]:
from my_module import *

external_func() # outputs -> 23

ModuleNotFoundError: No module named 'my_module'

In [41]:
_internal_func() # outputs -> NameError: "name '_internal_func' is not defined"

42

In [42]:
import my_module
my_module._internal_func()

ModuleNotFoundError: No module named 'my_module'

In [43]:
from my_module import _internal_func
_internal_func()

ModuleNotFoundError: No module named 'my_module'

In [44]:
a = Test()
a.some_method()

AttributeError: 'Test' object has no attribute 'some_method'

In [45]:
class Test2(Test):
  pass

In [46]:
b = Test2()
b.some_method()

AttributeError: 'Test2' object has no attribute 'some_method'

### Single Trailing Underscore: `var_`
Sometimes the most fitting name for a variable is already taken by a keyword. Therefore names like **class** or **def** cannot be used as variable names in Python. In this case you can append a single underscore to break the naming conflict:

In [47]:
def make_object(name, class):
  pass

SyntaxError: invalid syntax (ipython-input-665341856.py, line 1)

In [48]:
def make_object(name, class_):
    pass

### Double Leading Underscore: `__var`
The use of double underscore (__) in front of a name (specifically a method name) is not a convention; it has a specific meaning to the interpreter.

Python mangles these names and it is used to avoid name clashes with names defined by subclasses.

The interpreter changes the name of the variable in a way that makes it harder to create collisions when the class is extended later.

In [49]:
class Person:
  def __init__(self):
    self.name = 'Sarah'
    self._age = 26
    self.__id = 30

In [50]:
p = Person()
dir(p)

['_Person__id',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_age',
 'name']

In [51]:
print(p.name)
print(p._age)

Sarah
26


In [52]:
#p._Person__id
p.__id

AttributeError: 'Person' object has no attribute '__id'

In [53]:
p._Person__id

30

From the above list of object’s attributes, we can see `self.name` and `self._age` appears unchanged and behaves the same way.

 However,` __id` is mangled to `_Person__id`. This is the name mangling that the Python interpreter applies. It does this to protect the variable from getting overridden in subclasses.

 Let's look at another example:

In [54]:
class Test:
  def __init__(self):
    self.foo = 11
    self._bar = 23
    self.__baz = 23

In [55]:
t = Test()
dir(t)

['_Test__baz',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_bar',
 'foo']

If you look closely you’ll see there’s an attribute called `_Test__baz` on this object. This is the **name mangling** that the Python interpreter applies. It does this to protect the variable from getting overridden in subclasses.

Let’s create another class that extends the `Test` class and attempts to override its existing attributes added in the constructor:

In [56]:
class ExtendedTest(Test):
  def __init__(self):
    super().__init__() # self.foo, self._bar, self.__baz
    self.foo = 'overridden'
    self._bar = 'overridden'
    self.__baz = 'overridden'

In [57]:
t2 = ExtendedTest()
print(t2.foo)
print(t2._bar)
print(t2.__baz)

overridden
overridden


AttributeError: 'ExtendedTest' object has no attribute '__baz'

In [58]:
dir(t2)

['_ExtendedTest__baz',
 '_Test__baz',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_bar',
 'foo']

In [59]:
t2._ExtendedTest__baz

'overridden'

But the original _Test__baz is also still around:

In [60]:
t2._Test__baz

23

Double underscore name mangling is fully transparent to the programmer. Take a look at the following example that will confirm this:

In [61]:
class ManglingTest:
  def __init__(self):
    self.__mangled = 'hello'

  def get_mangled(self):
    return self.__mangled

In [62]:
ManglingTest().get_mangled() # ManglingTest().__mangled

'hello'

In [63]:
ManglingTest().__mangled

AttributeError: 'ManglingTest' object has no attribute '__mangled'

Name mangling also applies to method names. It affects all names that start with two underscore characters (“dunders”) in a class context:

In [64]:
class MangledMethod:
  def __method(self):
      return 42

  def call_it(self):
      return self.__method()

In [65]:
MangledMethod().__method()

AttributeError: 'MangledMethod' object has no attribute '__method'

In [66]:
MangledMethod().call_it()

42

Here’s another example of name mangling in action:

In [67]:
_MangledGlobal__mangled = 23

class MangledGlobal:

  def test(self):
    return __mangled

MangledGlobal().test()

23

In this example we declared a global variable called `_MangledGlobal__mangled`. Then we accessed the variable inside the context of a class named `MangledGlobal`. Because of name mangling we were able to reference the `_MangledGlobal__mangled` global variable as just `__mangled` inside the `test()` method on the class.

### Double Leading and Trailing Underscore: `__var__`

Name mangling is **not applied** if a name **starts and ends with double underscores**. Names that have leading and trailing double underscores (“dunders”) are reserved for special use like the `__init__` method for object constructors, or `__call__` method to make object callable. These methods are known as dunder methods.

This is just a convention, a way for the Python system to use names that won’t conflict with user-defined names. Therefore, dunders are just a convention and are left untouched by the Python interpreter.

In [68]:
class Person:
  """Some class"""
  def __init__(self):
    self.__name__ = 'Sarah'

Person().__name__

'Sarah'

In [69]:
p = Person()
p.__name__

'Sarah'

In [70]:
p.__doc__

'Some class'

In [71]:
dir(p)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__']

In [72]:
class Person:
  """Some class"""

  def __init__(self):
    self.__name__ = 'Sarah'

  def __call__(self, *args):
    for i in args:
      print(i)


In [73]:
p = Person()
p(1, 2, 5, 6, 5, 'sa')

1
2
5
6
5
sa


### Single Underscore: `_`
A single standalone underscore is sometimes used as a name to indicate that a variable is temporary or insignificant.

For example, in the following loop we don’t need access to the running index and we can use `_` to indicate that it is just a temporary value:

In [74]:
for _ in range(3):
  print('Barev')

Barev
Barev
Barev


You can also use single underscores in unpacking expressions to ignore particular values. Again, this meaning is “per convention” only and there’s no special behavior triggered in the Python interpreter. The single underscore is simply a valid variable name that’s sometimes used for this purpose.

In [75]:
car = ('red', 'auto', 12, 3812.4)
color, _, _, mileage = car

Besides its use as a temporary variable, “_” is a special variable in most Python coding evironments  that represents the result of the last expression evaluated by the interpreter.

In [76]:
print(_)


12


Here’s a quick summary of the five underscore patterns in Python:

| Pattern	| Example	| Meaning |
| ------ | ----- | ----- |
|Single Leading Underscore	| `_var` |	Naming convention indicating a name is meant for internal use. Generally not enforced by the Python interpreter (except in wildcard imports) and meant as a hint to the programmer only. |
|Single Trailing Underscore|	`var_`|	Used by convention to avoid naming conflicts with Python keywords. |
|Double Leading Underscore|`__var`|	Triggers name mangling when used in a class context. Enforced by the Python interpreter.|
|Double Leading and Trailing Underscore|	`__var__`	| Indicates special methods defined by the Python language. Avoid this naming scheme for your own attributes.|
|Single Underscore|	`_`	| Sometimes used as a name for temporary or insignificant variables (“don’t care”). Also: The result of the last expression in a Python environment.

# Instance, Static and Class Methods

So far we have worked with instance methods that take the object instance as the first argument of the function defined inside a class. There is also another option for coding simple functions associated with a class that
may be called through either the class or its instances. We can code
classes with `static` and `class` methods, neither of which requires an instance argument
to be passed in when invoked. To designate such methods, classes call the built-in
functions ``staticmethod`` and ``classmethod`` .


In [77]:
class Methods:

  def imeth(self, x): # Normal instance method: passed a self
    print([self, x])

  @staticmethod
  def smeth(x):
    print([x]) # Static: no instance passed
  @classmethod
  def cmeth(cls, x):
    print([cls, x]) # Class: gets class, not instance

#   smeth = staticmethod(smeth)
#   cmeth = classmethod(cmeth)

Notice how the last two assignments in this code simply reassign (a.k.a. rebind) the
method names `smeth` and `cmeth`. Attributes are created and changed by any assignment
in a class statement, so these final assignments simply overwrite the assignments made
earlier by the `def`s. As we’ll see later, the special `@` syntax works here as an alternative, but we will cover it in ``Decorators`` topic.

Technically, Python supports three kinds of class-related methods, with differing
argument protocols:

* ``Instance`` methods, passed a `self` instance object (the default)
* ``Static`` methods, passed no extra object (via `staticmethod` )
* ``Class`` methods, passed a ``class`` object (via `classmethod`)

`Instance` methods are the normal and default case that we’ve seen so far. An
`instance` method must always be called with an instance object. When you call it
through an instance, Python passes the instance to the first (leftmost) argument automatically; when you call it through a class, you must pass along the instance manually:

In [78]:
obj = Methods()
obj.imeth(1) # Methods.imeth(obj, 1)
Methods.imeth(obj, 2)

[<__main__.Methods object at 0x7d2759f08e90>, 1]
[<__main__.Methods object at 0x7d2759f08e90>, 2]


``Static`` methods are called without an instance argument. Unlike simple
functions outside a class, their names are local to the scopes of the classes in which they
are defined, and they may be looked up by inheritance.

In [79]:
Methods.smeth(3)
obj.smeth(4)

[3]
[4]


``Class`` methods are similar, but Python automatically passes the class (not an instance)
in to a class method’s first (leftmost) argument, whether it is called through a class or
an instance:

In [80]:
Methods.cmeth(5)
obj.cmeth(6)

[<class '__main__.Methods'>, 5]
[<class '__main__.Methods'>, 6]


There is also another type of methods called ``metaclass`` methods, which we will cover later.

One of the main uses of `classmethod` is to define alternative `constructors`:

In [81]:
class y:
  def __init__(self, astring):
    self.s = astring

  def from_list(cls, alist):
    x = cls('') # x.s = ''
    x.s = ','.join(str(s) for s in alist)
    return x

  def __repr__(self):
    return f'y({self.s})'

  from_list = classmethod(from_list)


In [82]:
y1 = y('xx')
y1

y(xx)

In [83]:
y2 = y.from_list(range(3))
y2

y(0,1,2)

Now if we subclass `y`, the `classmethod` keeps working

In [84]:
class k(y):
  def __repr__(self):
    return f'k({self.s.upper()})'

In [85]:
k1 = k.from_list(['za','bu'])
k1

k(ZA,BU)

Let's go over these concepts with an example:

In [86]:
class Pizza:
  def __init__(self, ingredients):
      self.ingredients = ingredients

  def __repr__(self):
      return f'Pizza({self.ingredients})'

In [87]:
Pizza(['cheese', 'tomatoes'])

Pizza(['cheese', 'tomatoes'])

We know that there are many types of pizza with different ingredients:

In [88]:
Pizza(['mozzarella', 'tomatoes'])

Pizza(['mozzarella', 'tomatoes'])

In [89]:
Pizza(['mozzarella', 'tomatoes', 'ham', 'mushrooms'])

Pizza(['mozzarella', 'tomatoes', 'ham', 'mushrooms'])

In [90]:
Pizza(['mozzarella'] * 4)

Pizza(['mozzarella', 'mozzarella', 'mozzarella', 'mozzarella'])

All of these pizzas have their names, so we can give the users of our Pizza class a better interface for creating the pizza objects they want. A nice and clean way to do that is by using class methods as factory functions for the different kinds of pizzas we can create

In [91]:
class Pizza:
  def __init__(self, ingredients):
      self.ingredients = ingredients

  def __repr__(self):
      return f'Pizza({self.ingredients})'

  def margherita(cls):
      return cls(['mozzarella', 'tomatoes'])

  def prosciutto(cls):
      return cls(['mozzarella', 'tomatoes', 'ham'])

  margherita = classmethod(margherita)
  prosciutto = classmethod(prosciutto)

Let's see how we can use these methods:

In [92]:
Pizza.margherita()

Pizza(['mozzarella', 'tomatoes'])

In [93]:
Pizza.prosciutto()

Pizza(['mozzarella', 'tomatoes', 'ham'])

As you can see, we can use the factory functions to create new Pizza objects that are configured the way we want them. They all use the same ``__init__`` constructor internally and simply provide a shortcut for remembering all of the various ingredients.

Another way to look at this use of class methods is that they allow you to define alternative constructors for your classes. Python only allows one ``__init__`` method per class. Using class methods it’s possible to add as many alternative constructors as necessary.

Now let's use static methods on this same example:

In [94]:
import math

class Pizza:
  def __init__(self, radius, ingredients):
    self.radius = radius
    self.ingredients = ingredients

  def __repr__(self):
    return (f'Pizza({self.radius}, '
            f'{self.ingredients})')

  def area(self):
    return self.circle_area(self.radius)

  def circle_area(r):
    return r ** 2 * math.pi

  circle_area = staticmethod(circle_area)

We modified the constructor and ``__repr__`` to accept an extra `radius` argument.

We also added an `area()` instance method that calculates and returns the pizza’s area.

Instead of calculating the area directly within `area()`, using the well-known circle area formula, we created a separate `circle_area()` static method.

In [95]:
p = Pizza(4, ['mozzarella', 'tomatoes'])

In [96]:
p.area()

50.26548245743669

In [97]:
Pizza.circle_area(4)

50.26548245743669

In [98]:
p.circle_area(4)

50.26548245743669

In [99]:
import math

def circle_area(r):
  return r ** 2 * math.pi


class Pizza:
  def __init__(self, radius, ingredients):
    self.radius = radius
    self.ingredients = ingredients

  def __repr__(self):
    return (f'Pizza({self.radius}, '
            f'{self.ingredients})')

  def area(self):
    return circle_area(self.radius)


In [100]:
p = Pizza(2, ['mozzarella', 'tomatoes'])
p.area()

12.566370614359172

In summary, `classmethods` make it possible to call the method on the class instead of an object.

* Instance methods need a class instance and can access the instance through `self`.
* Class methods don’t need a class instance. They can’t access the instance (`self`) but they have access to the class itself via `cls`.
* Static methods don’t have access to `cls` or `self`. They work like regular functions but belong to the class’s namespace.
* Static and class methods communicate and enforce developer intent about class design. This can have maintenance benefits.
